<a href="https://colab.research.google.com/github/sunnyskydream/Sequential-RAG-Marketing-Engine/blob/main/Sequential_RAG_Marketing_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**🚀 Sequential RAG Engine**

AI-powered personalization engine combining clickstream + demographic data for intent-aware marketing

📌 Overview
This project explores how marketing teams can move beyond static segmentation by combining user demographics with behavioral signals (clickstream data).

Using a Retrieval-Augmented Generation (RAG) approach, the system transforms raw user data into semantic understanding, enabling dynamic personalization across:
Email targeting,
Product recommendations,
Landing page personalization,
Campaign messaging.


🎯 Key Features\
🔍 Semantic user representation via embeddings\
⚡ Vector-based retrieval of user context\
✨ RAG-based personalized content generation\
🧪 Interactive prototyping via notebook environment\

🏗️ Architecture\
User Data (Clickstream + Demographics)\
        ↓\
Embedding Layer (OpenAI)\
        ↓\
Vector Database (Supabase)\
        ↓\
Retrieval (Relevant Context)\
        ↓\
RAG (Content Generation)

📊 Dataset\
Clickstream Data (Source: Kaggle E-commerce customer journey dataset)\
Demographic Data (Synthesized and joined via user_id
Simulates CDP-like user profiles)\

🛠️ Tech Stack\
Embeddings: OpenAI (text-embedding-3-small)\
Vector DB: Supabase\
Prototyping: Google Colab\
Language: Python

**Load Data**

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sufya6/e-commerce-customer-journey-click-to-conversion")

print("Path to dataset files:", path)

100%|██████████| 164k/164k [00:00<00:00, 54.5MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/sufya6/e-commerce-customer-journey-click-to-conversion/versions/1


In [ ]:
import pandas as pd
import numpy as np
import random

# 1. Simulate User_ID (In practice, please retrieve unique User_IDs from `clickstream_df`.)
user_ids = [f"USER_{i:04d}" for i in range(1, 501)]

def generate_demographics(ids):
    data = []
    income_levels = ['Low', 'Medium', 'High', 'Ultra High']
    interests = ['Gaming', 'DIY', 'Professional', 'Streaming', 'Esports']
    genders = ['Male', 'Female', 'Non-binary']

    for uid in ids:
        data.append({
            'user_id': uid,
            'age': random.randint(18, 55),
            'gender': random.choice(genders),
            'location': random.choice(['San Jose', 'San Francisco', 'New York', 'Austin', 'Seattle']),
            'annual_income': random.choice(income_levels),
            'loyalty_score': random.randint(1, 10),
            'primary_interest': random.choice(interests)
        })
    return pd.DataFrame(data)

# Generate Static Table
demographics_df = generate_demographics(user_ids)

# Save to CSV
demographics_df.to_csv('static_demographics.csv', index=False)
print("Static Demographics generated successfully with Gender!")
print(demographics_df.head())

Static Demographics generated successfully with Gender!
     user_id  age      gender       location annual_income  loyalty_score  \
0  USER_0001   22        Male         Austin           Low              5   
1  USER_0002   19  Non-binary       San Jose          High              7   
2  USER_0003   51        Male  San Francisco          High              5   
3  USER_0004   27  Non-binary        Seattle        Medium              8   
4  USER_0005   53      Female       San Jose           Low              3   

  primary_interest  
0          Esports  
1              DIY  
2     Professional  
3           Gaming  
4        Streaming  


**Semantic Stringification**\
Transforms each data column into human-readable sentences, making them easier to feed into an Embedding model.


In [ ]:
import os
import pandas as pd

# 1.Automatically detect CSV files in the directory
files = [f for f in os.listdir(path) if f.endswith('.csv')]
print("在目錄中找到的 CSV 檔案有：", files)

if not files:
    print("找不到任何 CSV 檔案，請檢查目錄內容。")
else:
    # 2. Select the first CSV file found
    actual_file_name = files[0]
    csv_path = os.path.join(path, actual_file_name)
    print(f"正在讀取檔案：{actual_file_name}")

    clickstream_df = pd.read_csv(csv_path)

    # 3. Check the field name (Kaggle datasets typically use 'Customer_ID' or 'User_ID')
    print("原始欄位名稱：", clickstream_df.columns.tolist())

    # Here, potential User ID fields are automatically renamed to 'user_id' to align with the subsequent code
    id_col = next((col for col in clickstream_df.columns if 'ID' in col.upper()), None)
    if id_col:
        clickstream_df.rename(columns={id_col: 'user_id'}, inplace=True)
        print(f"已將 '{id_col}' 改名為 'user_id'")

    print(clickstream_df.head())

在目錄中找到的 CSV 檔案有： ['customer_journey.csv']
正在讀取檔案：customer_journey.csv
原始欄位名稱： ['SessionID', 'UserID', 'Timestamp', 'PageType', 'DeviceType', 'Country', 'ReferralSource', 'TimeOnPage_seconds', 'ItemsInCart', 'Purchased']
已將 'SessionID' 改名為 'user_id'
     user_id     UserID            Timestamp      PageType DeviceType  \
0  session_0  user_2223  2025-01-20 22:53:34          home    Desktop   
1  session_1  user_2192  2025-02-26 12:57:10          home     Tablet   
2  session_1  user_2192  2025-02-26 12:59:11  product_page     Tablet   
3  session_2  user_1708  2025-06-24 15:40:46          home     Mobile   
4  session_3  user_2976  2025-06-11 07:21:02          home     Tablet   

   Country ReferralSource  TimeOnPage_seconds  ItemsInCart  Purchased  
0    India   Social Media                  55            0          0  
1  Germany          Email                  99            0          0  
2  Germany          Email                 121            0          0  
3    India         Googl

In [ ]:
import pandas as pd
import random

# 1. Extract the top 500 "real" user_ids from the clickstream
real_user_ids = clickstream_df['user_id'].unique()[:500]

# 2. Regenerate `demographics_df` using these real IDs.
def generate_real_demographics(ids):
    data = []
    income_levels = ['Low', 'Medium', 'High', 'Ultra High']
    interests = ['Gaming', 'DIY', 'Professional', 'Streaming', 'Esports']
    genders = ['Male', 'Female', 'Non-binary']

    for uid in ids:
        data.append({
            'user_id': uid, # Use Real ID
            'age': random.randint(18, 55),
            'gender': random.choice(genders),
            'location': random.choice(['San Jose', 'San Francisco', 'New York', 'Austin', 'Seattle']),
            'annual_income': random.choice(income_levels),
            'loyalty_score': random.randint(1, 10),
            'primary_interest': random.choice(interests)
        })
    return pd.DataFrame(data)

# Update static data tables
demographics_df = generate_real_demographics(real_user_ids)

# 3. Perform semantic merging again
test_df = clickstream_df[clickstream_df['user_id'].isin(real_user_ids)].head(100).copy()
test_df['semantic_context'] = test_df.apply(lambda row: stringify_user_context(row, demographics_df), axis=1)

# 4. Examination Results
success_count = test_df['semantic_context'].notnull().sum()
print(f"--- 成功生成 {success_count} 筆語義描述 ---")

if success_count > 0:
    print("\n驗證第一筆資料：")
    print(test_df['semantic_context'].dropna().iloc[1])

--- 成功生成 100 筆語義描述 ---

驗證第一筆資料：
這是一位住在 San Francisco 的 40 歲 Female，主要興趣是 Professional，年收入水平為 Ultra High。 目前正透過 Tablet 進行 進行特定操作，在該環節停留時間約為 2025-02-26 12:57:10 秒。 根據行為模式，該用戶正在初步探索產品資訊。


**Manual Validation on ChatGPT**

*case 1*\
prompt:
Role: 你是一位 Corsair 的資深數位行銷專家，擅長根據用戶行為進行精準投流與內容生成。 Task: 我將提供一段結合了「靜態人口統計」與「即時點擊流行為」的用戶描述。請根據這段描述，為 Corsair 生成以下三個行銷輸出： Email 主旨： 針對該用戶的購買意圖，寫一個高點擊率的主旨。
產品推薦： 根據其興趣與裝置，推薦一款 Corsair 的產品（如鍵盤、滑鼠、耳機、水冷）。
行動呼籲 (CTA)： 一句能轉化該用戶的短文案。
用戶語義描述： [這是一位住在 New York 的 54 歲 Female，主要興趣是 Professional，年收入水平為 High。 目前正透過 Desktop 進行 進行特定操作，在該環節停留時間約為 2025-01-20 22:53:34 秒。 根據行為模式，該用戶正在初步探索產品資訊]

response:
以下是根據該用戶輪廓（高收入、專業導向、初期探索階段、Desktop 使用）所設計的精準行銷輸出：

📩 Email 主旨\
Elevate Your Workspace: Premium Performance Meets Professional Precision

🖥️ 產品推薦\
Corsair K100 RGB Mechanical Gaming Keyboard（高階機械鍵盤）\
理由：
對於 Professional 高收入族群，鍵盤不只是娛樂，更是「效率與質感」的象徵
超高速輸入與高度客製化（適合長時間工作 + 偶爾娛樂）
高端鋁製設計，符合成熟用戶對品質與外觀的要求

🎯 行動呼籲 (CTA)\
Upgrade your desk to a performance powerhouse—discover the difference today.

*case 2*\
prompt:
Role: 你是一位 Corsair 的資深數位行銷專家，擅長根據用戶行為進行精準投流與內容生成。 Task: 我將提供一段結合了「靜態人口統計」與「即時點擊流行為」的用戶描述。請根據這段描述，為 Corsair 生成以下三個行銷輸出： Email 主旨： 針對該用戶的購買意圖，寫一個高點擊率的主旨。 產品推薦： 根據其興趣與裝置，推薦一款 Corsair 的產品（如鍵盤、滑鼠、耳機、水冷）。 行動呼籲 (CTA)： 一句能轉化該用戶的短文案。 用戶語義描述：[這是一位住在 San Francisco 的 40 歲 Female，主要興趣是 Professional，年收入水平為 Ultra High。 目前正透過 Tablet 進行 進行特定操作，在該環節停留時間約為 2025-02-26 12:57:10 秒。 根據行為模式，該用戶正在初步探索產品資訊]


reponse:
以下針對該用戶（San Francisco、40歲女性、Ultra High 收入、Professional 導向、Tablet 使用、初步探索階段）設計的精準行銷輸出：

📩 Email 主旨\
Refine Your Digital Experience—Luxury Performance, Designed for You

🎧 產品推薦\
Corsair Virtuoso RGB Wireless XT Headset（高階無線耳機）
理由：
符合 Ultra High 收入族群對「質感 + 功能」的雙重要求（高級材質 + 極致音質）
適合 Professional 使用場景：遠距會議、商務通話、沉浸式專注工作
支援多裝置（包含 Tablet），無縫切換工作與生活
低調奢華設計，貼合成熟專業女性審美\
🎯 行動呼籲 (CTA)\
Experience uncompromising clarity—upgrade your audio, wherever work takes you.

**Batch Embedding and upload to Supabase**

1. get configuration\
OPENAI_API_KEY = "your_OPENAI_API_KEY"\
SUPABASE_URL = "your_PROJECT_URL"\
SUPABASE_KEY = "your_SERVICE_ROLE_KEY"\

2. SUPABASE set up\
-- Step 1: Enable the pgvector extension
CREATE EXTENSION IF NOT EXISTS vector;

  -- Step 2: Create the data table\
CREATE TABLE IF NOT EXISTS user_context_vectors (\
  id bigserial PRIMARY KEY,\
  user_id text,\
  content text,\
  embedding vector(1536),\
  created_at timestamp with time zone DEFAULT now()\
);

In [ ]:
!pip install supabase openai

import openai
from supabase import create_client
import pandas as pd
import time

# --- 1. get configuration  ---
OPENAI_API_KEY = "your_OPENAI_API_KEY"
SUPABASE_URL = "your_PROJECT_URL"
SUPABASE_KEY = "your_SERVICE_ROLE_KEY"

# 初始化客戶端
openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

def get_embedding(text):
    """Converts semantic text into a 1536-dimensional vector."""
    try:
        response = openai_client.embeddings.create(
            input=text,
            model="text-embedding-3-small"
        )
        return response.data[0].embedding
    except Exception as e:
        print(f"❌ OpenAI Embedding 出錯: {e}")
        return None

# --- 2. 執行批次上傳 ---
print("🚀 啟動 Sequential-RAG-Engine: 正在將用戶意圖向量化並上傳至雲端...")

# We will use the previously generated `test_df`, which should already contain the 'semantic_context' column.
for index, row in test_df.iterrows():
    context_text = row['semantic_context']
    uid = row['user_id']

    if pd.isna(context_text):
        continue

    # Call AI to Convert to Vector
    vector_coords = get_embedding(context_text)

    if vector_coords:
        # Preparing to write to Supabasee
        payload = {
            "user_id": uid,
            "content": context_text,
            "embedding": vector_coords # This will be automatically stored in the `vector(1536)` field.
        }

        try:
            supabase.table("user_context_vectors").insert(payload).execute()
            print(f"✅ [成功] 用戶 {uid} 的語義向量已存入 Supabase")
        except Exception as e:
            print(f"❌ [失敗] 用戶 {uid} 存入資料庫時出錯: {e}")

    # Pause briefly to avoid triggering OpenAI's rate limits.
    time.sleep(0.2)

print("\n✨ 任務完成！你的點擊流數據現在已經變成具備『語義搜索』能力的向量資料庫了。")

🚀 啟動 Sequential-RAG-Engine: 正在將用戶意圖向量化並上傳至雲端...
✅ [成功] 用戶 session_0 的語義向量已存入 Supabase
✅ [成功] 用戶 session_1 的語義向量已存入 Supabase
✅ [成功] 用戶 session_1 的語義向量已存入 Supabase
✅ [成功] 用戶 session_2 的語義向量已存入 Supabase
✅ [成功] 用戶 session_3 的語義向量已存入 Supabase
✅ [成功] 用戶 session_4 的語義向量已存入 Supabase
✅ [成功] 用戶 session_4 的語義向量已存入 Supabase
✅ [成功] 用戶 session_5 的語義向量已存入 Supabase
✅ [成功] 用戶 session_5 的語義向量已存入 Supabase
✅ [成功] 用戶 session_5 的語義向量已存入 Supabase
✅ [成功] 用戶 session_5 的語義向量已存入 Supabase
✅ [成功] 用戶 session_5 的語義向量已存入 Supabase
✅ [成功] 用戶 session_6 的語義向量已存入 Supabase
✅ [成功] 用戶 session_7 的語義向量已存入 Supabase
✅ [成功] 用戶 session_7 的語義向量已存入 Supabase
✅ [成功] 用戶 session_8 的語義向量已存入 Supabase
✅ [成功] 用戶 session_8 的語義向量已存入 Supabase
✅ [成功] 用戶 session_9 的語義向量已存入 Supabase
✅ [成功] 用戶 session_9 的語義向量已存入 Supabase
✅ [成功] 用戶 session_9 的語義向量已存入 Supabase
✅ [成功] 用戶 session_10 的語義向量已存入 Supabase
✅ [成功] 用戶 session_10 的語義向量已存入 Supabase
✅ [成功] 用戶 session_11 的語義向量已存入 Supabase
✅ [成功] 用戶 session_11 的語義向量已存入 Supabase
✅ [成功] 用戶 session_11 的語義向量已存入 Supaba

**Simulated Real-time Retrieval**\
We will feed the system a "new user behavior" to see if it can retrieve the most relevant historical context from Supabase and prompt the AI ​​to generate precise marketing recommendations.

New Scenario: A high-income user from New York is currently browsing gaming headsets on a desktop computer and has remained on the page for over two minutes (demonstrating a high level of interest).

In [ ]:
def semantic_search(query_text, limit=1):
    # 1. Convert test questions into vectors.
    query_vector = get_embedding(query_text)

    # 2. Call Supabase RPC (Remote Procedure Call) for vector comparison.
    # Note: We first need to create a search function in the SQL Editor.
    rpc_input = {
        'query_embedding': query_vector,
        'match_threshold': 0.5, # Similarity Threshold
        'match_count': limit
    }

    # Use the simplest query method to verify whether the data can be retrieved.
    res = supabase.table("user_context_vectors").select("*").execute()
    return res.data[0]

# Simulate New User Behavior
test_query = "一位住在紐約的高收入玩家，正在瀏覽高階耳機，停留時間很長，看起來非常有購買意圖。"
result = semantic_search(test_query)

print("--- 檢索測試結果 ---")
print(f"系統匹配到的最相關歷史上下文：\n{result['content']}")

--- 檢索測試結果 ---
系統匹配到的最相關歷史上下文：
這是一位住在 San Francisco 的 44 歲 Female，主要興趣是 Gaming，年收入水平為 High。 目前正透過 Desktop 進行 進行特定操作，在該環節停留時間約為 2025-01-20 22:53:34 秒。 根據行為模式，該用戶正在初步探索產品資訊。


**Generate intent-aware personalized marketing content**

In [ ]:
def generate_marketing_advice(context):
    response = openai_client.chat.completions.create(
        model="gpt-4o", # or gpt-3.5-turbo
        messages=[
            {"role": "system", "content": "你是一位 Corsair 的資深行銷專家。"},
            {"role": "user", "content": f"根據以下這位相似用戶的背景：\n{context}\n\n請為這位用戶設計一個個性化的 Corsair 產品推薦理由，並寫出一句吸引人的 Email 主旨。"}
        ]
    )
    return response.choices[0].message.content

# Pass in the results just retrieved.
final_advice = generate_marketing_advice(result['content'])
print("\n--- Corsair AI 行銷建議 ---")
print(final_advice)


--- Corsair AI 行銷建議 ---
Email 主旨：解鎖終極遊戲體驗，專為您的風格量身打造！

產品推薦理由：

親愛的玩家，

作為一位位於 San Francisco 的資深遊戲玩家，您對遊戲的熱愛和高收入水平讓我們深信，您追求的不僅是娛樂，更是極致的遊戲體驗。Corsair 了解每一位玩家的獨特需求，因此我們為您精選了一系列頂尖產品，以助您在遊戲世界中大放異彩。

推薦的產品包括我們最新的 Corsair K100 RGB 鍵盤，憑藉其卓越的性能和時尚的設計，讓您在每一次按鍵中感受精確和舒適。此外，Corsair Virtuoso RGB Wireless 耳機以其卓越的音質和長時續航，讓您全心投入遊戲世界，不留遺憾。最後，我們的 Corsair iCUE SP120 RGB Pro 風扇，讓您的遊戲機箱瞬間煥發活力，以絢麗燈效製造沉浸式環境。

無論是性能還是設計，我們都力求完美，與您共同創造獨一無二的遊戲體驗。立即探索 Corsair 的產品，提升您的遊戲水平，成為真正無懈可擊的戰士！

祝遊戲愉快！  
Corsair 團隊
